## Simulación de un PID de forma iterativa

Librerías necesarias

In [ ]:
import matplotlib.pyplot as plt 

El Proceso a controlar viene descrito por el siguiente modelo discreto obtenido a un período de muestreo, T = 0.1 seg.

$$G_p(z)=\frac{0.007(z+0.94)}{(z-0.95)(z-0.86)}$$

Primero simularemos el comportamiento del proceso sin regulador en lazo cerrado

Expresamos dicha función de transferencia en la ecuación en diferencias equivalente:

$$y(k)=1.81y(k-1)-0.817y(k-2)+0.007u(k-1)+0.0066u(k-2)$$

Definimos las variables que van a almacenar los datos de la simulación así como otros parámetros:

In [ ]:
tsim = 15                  # Tiempo de simulación en segundos
T = 0.1                     # Período de muestreo
numiter = int(tsim/T)-2     # Número de iteraciones para la simulación
u = 1                       # Entrada escalón unitario
entrada = []
error = []
errorc = []
accont = []
salida = []
salcont = []

La función 'simula()' obtiene los valores de la simulación para instantes despúes del retardo

In [ ]:
def simula():
    global ym1, ym2, em1, em2
    for i in range(numiter):
        y = (1.81*ym1)-(0.817*ym2)+(0.007*em1)+(0.0066*em2)
        e = u - ym1
        entrada.append(u)
        error.append(e)
        salida.append(y)

        # Reasignamos las variables
        em2 = em1
        em1 = e
        ym2 = ym1
        ym1 = y

    # print (error)
    # print (salida)

Calculamos previamente los dos primeros valores ya que provocarían índices negativos en el bucle al calcular y(k)

In [ ]:
# Iteración k = 0
ek = 0
yk = 0
entrada.append(u)
error.append(ek)
salida.append(yk)

# Iteración k = 1
ek1 = u - yk
yk1 = (1.81*yk)+(0.007*ek1)
entrada.append(u)
error.append(ek1)
salida.append(yk1)

# Iteración k = 2
ek2 = u - yk1
yk2 = (1.81*yk1)-(0.817*yk)+(0.007*ek2)+(0.0066*ek1)
entrada.append(u)
error.append(ek2)
salida.append(yk2)

# Resto de iteraciones
ym1 = yk2
ym2 = yk1
em1 = ek2
em2 = ek1

simula()


Realizamos la obtención del resto de valores mediante un bucle:

In [ ]:
# Representar los datos
longitud_datos = len(salida)
ejex = []
for i in range(0, longitud_datos):
    ejex.append(i)
etiqueta = 'Salida'
# print(len(datos))
# print(len(ejex))

#plt.figure(figsize=(12, 8)) # Opcional: Define el tamaño de la figura
plt.plot(ejex, salida, label=etiqueta, color='blue', linestyle='solid') # Dibuja la línea
plt.plot(ejex, entrada, label='Referencia', color='red', linestyle='solid') # Dibuja la línea
# 3. Personalizar (opcional)
plt.title('Sistema sin controlar')
plt.xlabel(f'Muestras (T = {T} seg.)')
plt.ylabel('y(k)')
plt.legend() # Muestra la leyenda
plt.grid(True) # Añade una cuadrícula

# 4. Mostrar la gráfica
plt.show()        


Procedemos ahora a simular el sistema controlado con el regulador PID calculado:

$$G_R(z)=18.86\frac{(z-0.967)(z-0.7)}{z(z-1)}

Expresamos la salida del regulador (acción de control) mediante su correspondiente ecuación en diferencias:

$$u(k)=u(k-1)+18.86e(k)-31.4396e(k-1)+12.7663e(k-2)$$

In [ ]:
def sim_control():
    global ym1, ym2, em1, em2, acm1, acm2, e, u
    for i in range(numiter):
        ec = u - ym1
        c = acm1+(18.86*ec)-(31.4396*em1)+(12.7663*em2)
        y = (1.81*ym1)-(0.817*ym2)+(0.007*acm1)+(0.0066*acm2)
        #entrada.append(u)
        errorc.append(ec)
        accont.append(c)
        salcont.append(y)

        # Reasignamos las variables
        em2 = em1
        em1 = ec
        acm2 = acm1
        acm1 = c
        ym2 = ym1
        ym1 = y

    # print (error)
    # print (salida)
    

Calculamos los dos primeros valores de la simulación para evitar índices negativos en el bucle:

In [ ]:
# Iteración k = 0
u = 1       # Entrada escalón unitario
ek = 0
ac = 0
yk = 0
#entrada.append(u)
errorc.append(ek)
accont.append(ac)
salcont.append(yk)

# Iteración k = 1
ek1 = u - yk
ac1 = 18.86*ek1
yk1 = (1.81*yk)+(0.007*ac1)
#entrada.append(u)
errorc.append(ek1)
accont.append(ac1)
salcont.append(yk1)

# Iteración k = 2
ek2 = u - yk1
ac2 = ac1+(18.86*ek2)-(31.4396*ek1)+(12.7663*ek)
yk2 = (1.81*yk1)-(0.817*yk)+(0.007*ac2)+(0.0066*ac1)
#entrada.append(u)
errorc.append(ek2)
accont.append(ac2)
salcont.append(yk2)

ym1 = yk2
ym2 = yk1
em1 = ek2
em2 = ek1
acm1 = ac2
acm2 = ac1
ec = ek2

sim_control()

print(f'Sobreoscilación máxima: ',max(salcont))


In [ ]:
# Representar los datos
longitud_datos = len(salcont)
ejex = []
for i in range(0, longitud_datos):
    ejex.append(i)

# print(len(datos))
# print(len(ejex))
#plt.clf
plt.figure(figsize=(12, 8)) # Opcional: Define el tamaño de la figura
plt.plot(ejex, salcont, label='Salida controlada', color='blue', linestyle='solid') # Dibuja la línea
plt.plot(ejex, entrada, label='Referncia', color='red', linestyle='solid') # Dibuja la línea
# 3. Personalizar (opcional)
plt.title('Control PID')
plt.xlabel(f'Muestras (T = {T}seg.)')
plt.ylabel('y(k)')
plt.legend() # Muestra la leyenda
plt.grid(True) # Añade una cuadrícula

# 4. Mostrar la gráfica
plt.show()        


In [ ]:
# Representar los datos
longitud_datos = len(errorc)
ejex = []
for i in range(0, longitud_datos):
    ejex.append(i)

# print(len(datos))
# print(len(ejex))
#plt.clf
plt.figure(figsize=(12, 8)) # Opcional: Define el tamaño de la figura
plt.plot(ejex, errorc, label='Error', color='blue', linestyle='solid') # Dibuja la línea
plt.plot(ejex, accont, label='Señal de control', color='red', linestyle='solid') # Dibuja la línea
# 3. Personalizar (opcional)
plt.title('Control PID')
plt.xlabel(f'Muestras (T = {T}seg.)')
plt.ylabel('y(k)')
plt.legend() # Muestra la leyenda
plt.grid(True) # Añade una cuadrícula

# 4. Mostrar la gráfica
plt.show()        


In [ ]:
# Representar los datos
longitud_datos = len(salida)
ejex = []
for i in range(0, longitud_datos):
    ejex.append(i)

# print(len(datos))
# print(len(ejex))
#plt.clf
plt.figure(figsize=(12, 8)) # Opcional: Define el tamaño de la figura
plt.plot(ejex, salida, label='Salida sin PID', color='blue', linestyle='solid') # Dibuja la línea
plt.plot(ejex, salcont, label='Salida con PID', color='green', linestyle='solid') # Dibuja la línea
plt.plot(ejex, entrada, label='Referencia', color='red', linestyle='solid')
# 3. Personalizar (opcional)
plt.title('Control PID')
plt.xlabel(f'Muestras (T = {T}seg.)')
plt.ylabel('y(k)')
plt.legend() # Muestra la leyenda
plt.grid(True) # Añade una cuadrícula

# 4. Mostrar la gráfica
plt.show()        
